# Sarai experiment matrix

Runs ASR × diarization × LLM on one recording and writes a comparison report.
Wraps `scripts/run_experiment_matrix.py`; finished stages are skipped, so a re-run resumes.

**Kernel:** the `sarai` conda env. First time only, in a terminal:
```
conda activate sarai && pip install ipykernel && python -m ipykernel install --user --name sarai
```
**Before running:** stop `make dev` (the worker holds GPU memory).

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
# ffmpeg lives in the conda env's bin; make sure subprocesses find it.
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]
print("repo:", REPO, "| python:", sys.executable)

## 1. Settings
Edit these. `SOURCE` can be a video or audio file (any format ffmpeg reads).

In [ ]:
SOURCE = Path("/mnt/c/Users/champ/Downloads/COLAB SCGxMUICT Update Regular meeting-20261109_210541-Meeting Recording.mp4")
TITLE = "COLAB SCGxMUICT Update Regular meeting 2026-11-09"
OUT = REPO / "data/experiments" / "my-run"      # results land here

GEMINI_MODEL = "gemini-3.5-flash"   # gemini-3.8-flash was returning 503 "high demand"
GPU_LABEL = "RTX 5070 12GB"         # shown in the report only
OFFLINE = True                      # use cached models only; set False to allow downloads

## 2. Check GPU is free
If memory used is high, stop `make dev` / other workers first.

In [ ]:
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
                      "--format=csv"], capture_output=True, text=True).stdout)

## 3. Convert to 16 kHz mono WAV
Same ffmpeg path the app uses (`sarai.audio.normalize`).

In [ ]:
from sarai.audio import normalize, probe

OUT.mkdir(parents=True, exist_ok=True)
WAV = OUT / "audio.wav"
if not WAV.exists():
    normalize(SOURCE, WAV)
info = probe(WAV)
print(WAV, info)

## 4. Run the matrix
Order: diarize (pyannote) → ASR (large-v3, turbo) → LLM (deepseek, gemini) → report.
Output streams below. Interrupt the kernel to stop; run the cell again to resume.

In [ ]:
env = {**os.environ, "GEMINI_MODEL": GEMINI_MODEL}
if OFFLINE:
    env["HF_HUB_OFFLINE"] = "1"
cmd = [sys.executable, "scripts/run_experiment_matrix.py", "--wav", str(WAV), "--out", str(OUT),
       "--title", TITLE, "--gpu", GPU_LABEL, "all"]
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in proc.stdout:
        # Keep the stream readable: matrix progress, warnings and errors only.
        if " matrix " in line or "Error" in line or "Traceback" in line:
            print(line, end="")
finally:
    proc.wait()
print("exit code", proc.returncode)

## 5. Results: running time per experiment

In [ ]:
from IPython.display import HTML, display

def mmss(s):
    return "-" if s is None else f"{int(s // 60)}m{int(s % 60):02d}s"

rows = []
for r in sorted((OUT / "runs").glob("*/result.json")):
    d = json.loads(r.read_text())
    t = next((x for x in json.loads((OUT / "timings.json").read_text()) if x["run"] == d["run"]), {}) \
        if (OUT / "timings.json").exists() else {}
    rows.append(f"<tr><td>{d['run']}</td><td>{d['asr'].split('/')[-1]}</td><td>{d['diarization'].split('/')[-1]}</td>"
                f"<td>{d.get('llm_model', d['llm_provider'])}</td><td>{mmss(t.get('total_seconds'))}</td>"
                f"<td>{d.get('action_items', '-')}</td><td>{'FAILED' if d.get('error') else 'ok'}</td></tr>")
display(HTML("<table><tr><th>#</th><th>ASR</th><th>Diarization</th><th>LLM</th><th>Total time</th>"
             "<th>Actions</th><th>Status</th></tr>" + "".join(rows) + "</table>"))
print("report:", OUT / "report.pdf")

## 6. Re-run one stage
Delete its output, then run section 4 again. Examples:

In [ ]:
import shutil
# Redo all Gemini summaries:
# for d in (OUT / "runs").glob("*gemini*"): shutil.rmtree(d)
# Redo turbo transcripts:
# for f in (OUT / "transcripts").glob("turbo__*.json"): f.unlink()
# Rebuild only the report:
# subprocess.run([sys.executable, "scripts/run_experiment_matrix.py", "--wav", str(WAV), "--out", str(OUT), "--title", TITLE, "--gpu", GPU_LABEL, "report"], check=True)